In [31]:
import torch
gpu = "0"
device = torch.device("cuda:{}".format(gpu) if torch.cuda.is_available() else "cpu")

In [2]:
import os
import subprocess

def download_file(filepath: str, url: str):
    """Download file from URL using curl if it doesn't already exist."""
    if not os.path.exists(filepath):
        os.makedirs(os.path.dirname(filepath), exist_ok=True)
        print(f"Downloading to {filepath}...")
        subprocess.run(["curl", "-L", "-o", filepath, url], check=True)
    else:
        print(f"File already exists: {filepath}")



In [3]:
target_path = "yolo/darknet.py"
source_url = "https://raw.githubusercontent.com/ayooshkathuria/YOLO_v3_tutorial_from_scratch/master/darknet.py"
download_file(target_path, source_url)

In [4]:
target_path = "yolo/util.py"
source_url = "https://raw.githubusercontent.com/ayooshkathuria/YOLO_v3_tutorial_from_scratch/master/util.py"
download_file(target_path, source_url)

In [5]:
# Example usage
download_file("yolo/yolov3-weights.zip",
    "https://www.kaggle.com/api/v1/datasets/download/shivam316/yolov3-weights"
)

In [27]:
target_path = "yolo/dog-cycle-car.png"
source_url = "https://github.com/ayooshkathuria/pytorch-yolo-v3/raw/master/dog-cycle-car.png"
download_file(target_path, source_url)

In [28]:
target_path = "yolo/cfg/yolov3.cfg"
source_url = "https://raw.githubusercontent.com/ayooshkathuria/YOLO_v3_tutorial_from_scratch/master/cfg/yolov3.cfg"
download_file(target_path, source_url)

File already exists: yolo/cfg/yolov3.cfg


In [8]:
import zipfile

def extract_zip_file(zip_path: str, extract_path: str):
    """Extract ZIP file to target path if it exists."""
    if os.path.exists(zip_path):
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            zip_ref.extractall(extract_path)
        print(f"✅ Dataset extracted to: {extract_path}")
    else:
        print(f"⚠️ ZIP file not found: {zip_path}")



In [29]:
zip_path = "yolo/yolov3-weights.zip"
extract_path = "yolo"
extract_zip_file(zip_path, extract_path)

✅ Dataset extracted to: yolo


In [15]:
import sys
import os

yolo_path = os.path.abspath("yolo")
if yolo_path not in sys.path:
    sys.path.insert(0, yolo_path)

import darknet  # Now this works

In [23]:
base_dir = os.path.abspath(".")
print(base_dir)
cfg_path = os.path.join(base_dir, "yolo", "cfg", "yolov3.cfg")


blocks = darknet.parse_cfg(cfg_path)


print(darknet.create_modules(blocks))

/content
({'type': 'net', 'batch': '1', 'subdivisions': '1', 'width': '416', 'height': '416', 'channels': '3', 'momentum': '0.9', 'decay': '0.0005', 'angle': '0', 'saturation': '1.5', 'exposure': '1.5', 'hue': '.1', 'learning_rate': '0.001', 'burn_in': '1000', 'max_batches': '500200', 'policy': 'steps', 'steps': '400000,450000', 'scales': '.1,.1'}, ModuleList(
  (0): Sequential(
    (conv_0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
    (batch_norm_0): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (leaky_0): LeakyReLU(negative_slope=0.1, inplace=True)
  )
  (1): Sequential(
    (conv_1): Conv2d(32, 64, kernel_size=(3, 3), stride=(2, 2), padding=(1, 1), bias=False)
    (batch_norm_1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (leaky_1): LeakyReLU(negative_slope=0.1, inplace=True)
  )
  (2): Sequential(
    (conv_2): Conv2d(64, 32, kernel_size=(1, 1), stride=(1, 1), bias=Fa

In [24]:
def predict_transform2(prediction, inp_dim, anchors, num_classes):
    device = prediction.device  # Automatically align all tensors to this device

    batch_size = prediction.size(0)
    stride = inp_dim // prediction.size(2)
    grid_size = inp_dim // stride
    bbox_attrs = 5 + num_classes
    num_anchors = len(anchors)

    prediction = prediction.view(batch_size, bbox_attrs * num_anchors, grid_size * grid_size)
    prediction = prediction.transpose(1, 2).contiguous()
    prediction = prediction.view(batch_size, grid_size * grid_size * num_anchors, bbox_attrs)

    anchors = [(a[0] / stride, a[1] / stride) for a in anchors]

    # Sigmoid center x, y and object confidence
    prediction[:, :, 0] = torch.sigmoid(prediction[:, :, 0])
    prediction[:, :, 1] = torch.sigmoid(prediction[:, :, 1])
    prediction[:, :, 4] = torch.sigmoid(prediction[:, :, 4])

    # Center offsets
    grid = torch.arange(grid_size, device=device)
    a, b = torch.meshgrid(grid, grid, indexing='ij')
    x_offset = a.contiguous().view(-1, 1).float()
    y_offset = b.contiguous().view(-1, 1).float()
    x_y_offset = torch.cat((x_offset, y_offset), 1).repeat(1, num_anchors).view(-1, 2).unsqueeze(0)

    prediction[:, :, :2] += x_y_offset

    # Log-space transform height and width
    anchors = torch.tensor(anchors, dtype=torch.float32, device=device)
    anchors = anchors.repeat(grid_size * grid_size, 1).unsqueeze(0)
    prediction[:, :, 2:4] = torch.exp(prediction[:, :, 2:4]) * anchors

    # Class scores
    prediction[:, :, 5:5 + num_classes] = torch.sigmoid(prediction[:, :, 5:5 + num_classes])

    # Scale boxes to original image size
    prediction[:, :, :4] *= stride

    return prediction

In [25]:
from util import *

class MyDarknet(nn.Module):
    def __init__(self, cfgfile):
        super(MyDarknet, self).__init__()
        # load the config file and create our model
        self.blocks = darknet.parse_cfg(cfgfile)
        self.net_info, self.module_list = darknet.create_modules(self.blocks)

    def forward(self, x):
        modules = self.blocks[1:]
        outputs = {}   #We cache the outputs for the route layer

        write = 0
        # run forward propagation. Follow the instruction from dictionary modules
        for i, module in enumerate(modules):
            module_type = (module["type"])

            if module_type == "convolutional" or module_type == "upsample":
                # do convolutional network
                x = self.module_list[i](x)

            elif module_type == "route":
                # concat layers
                layers = module["layers"]
                layers = [int(a) for a in layers]

                if (layers[0]) > 0:
                    layers[0] = layers[0] - i

                if len(layers) == 1:
                    x = outputs[i + (layers[0])]

                else:
                    if (layers[1]) > 0:
                        layers[1] = layers[1] - i

                    map1 = outputs[i + layers[0]]
                    map2 = outputs[i + layers[1]]
                    x = torch.cat((map1, map2), 1)


            elif  module_type == "shortcut":
                from_ = int(module["from"])
                # residual network
                x = outputs[i-1] + outputs[i+from_]

            elif module_type == 'yolo':
                anchors = self.module_list[i][0].anchors
                #Get the input dimensions
                inp_dim = int (self.net_info["height"])

                #Get the number of classes
                num_classes = int (module["classes"])

                #Transform
                # predict_transform is in util.py
                batch_size = x.size(0)
                stride =  inp_dim // x.size(2)
                grid_size = inp_dim // stride
                bbox_attrs = 5 + num_classes
                num_anchors = len(anchors)

                x = predict_transform2(x, inp_dim, anchors, num_classes)
                if not write:              #if no collector has been intialised.
                    detections = x
                    write = 1

                else:
                    detections = torch.cat((detections, x), 1)

            outputs[i] = x

        return detections


    def load_weights(self, weightfile):
        '''
        Load pretrained weight
        '''
        #Open the weights file
        fp = open(weightfile, "rb")

        #The first 5 values are header information
        # 1. Major version number
        # 2. Minor Version Number
        # 3. Subversion number
        # 4,5. Images seen by the network (during training)
        header = np.fromfile(fp, dtype = np.int32, count = 5)
        self.header = torch.from_numpy(header)
        self.seen = self.header[3]

        weights = np.fromfile(fp, dtype = np.float32)

        ptr = 0
        for i in range(len(self.module_list)):
            module_type = self.blocks[i + 1]["type"]

            #If module_type is convolutional load weights
            #Otherwise ignore.

            if module_type == "convolutional":
                model = self.module_list[i]
                try:
                    batch_normalize = int(self.blocks[i+1]["batch_normalize"])
                except:
                    batch_normalize = 0

                conv = model[0]


                if (batch_normalize):
                    bn = model[1]

                    #Get the number of weights of Batch Norm Layer
                    num_bn_biases = bn.bias.numel()

                    #Load the weights
                    bn_biases = torch.from_numpy(weights[ptr:ptr + num_bn_biases])
                    ptr += num_bn_biases

                    bn_weights = torch.from_numpy(weights[ptr: ptr + num_bn_biases])
                    ptr  += num_bn_biases

                    bn_running_mean = torch.from_numpy(weights[ptr: ptr + num_bn_biases])
                    ptr  += num_bn_biases

                    bn_running_var = torch.from_numpy(weights[ptr: ptr + num_bn_biases])
                    ptr  += num_bn_biases

                    #Cast the loaded weights into dims of model weights.
                    bn_biases = bn_biases.view_as(bn.bias.data)
                    bn_weights = bn_weights.view_as(bn.weight.data)
                    bn_running_mean = bn_running_mean.view_as(bn.running_mean)
                    bn_running_var = bn_running_var.view_as(bn.running_var)

                    #Copy the data to model
                    bn.bias.data.copy_(bn_biases)
                    bn.weight.data.copy_(bn_weights)
                    bn.running_mean.copy_(bn_running_mean)
                    bn.running_var.copy_(bn_running_var)

                else:
                    #Number of biases
                    num_biases = conv.bias.numel()

                    #Load the weights
                    conv_biases = torch.from_numpy(weights[ptr: ptr + num_biases])
                    ptr = ptr + num_biases

                    #reshape the loaded weights according to the dims of the model weights
                    conv_biases = conv_biases.view_as(conv.bias.data)

                    #Finally copy the data
                    conv.bias.data.copy_(conv_biases)

                #Let us load the weights for the Convolutional layers
                num_weights = conv.weight.numel()

                #Do the same as above for weights
                conv_weights = torch.from_numpy(weights[ptr:ptr+num_weights])
                ptr = ptr + num_weights

                conv_weights = conv_weights.view_as(conv.weight.data)
                conv.weight.data.copy_(conv_weights)

In [26]:
import cv2
import torch

def get_test_input():
    img_path=os.path.join(base_dir, "yolo", "dog-cycle-car.png")
    img = cv2.imread(img_path)
    img = cv2.resize(img, (416,416))          #Resize to the input dimension
    img_ =  img[:,:,::-1].transpose((2,0,1))  # BGR -> RGB | H X W C -> C X H X W
    img_ = img_[np.newaxis,:,:,:]/255.0       #Add a channel at 0 (for batch) | Normalise
    img_ = torch.from_numpy(img_).float()     #Convert to float
    img_ = Variable(img_)                     # Convert to Variable
    return img_

In [32]:
from util import *

model = MyDarknet(cfg_path).to(device)
inp = get_test_input().to(device)
pred = model(inp)
print (pred)

tensor([[[1.6719e+01, 1.4162e+01, 1.0237e+02,  ..., 5.0574e-01,
          4.7884e-01, 5.9145e-01],
         [1.3972e+01, 1.3669e+01, 2.3103e+02,  ..., 5.0696e-01,
          5.2021e-01, 5.0196e-01],
         [1.4981e+01, 1.9572e+01, 5.3554e+02,  ..., 5.3606e-01,
          4.0815e-01, 5.5034e-01],
         ...,
         [4.1160e+02, 4.1181e+02, 7.9593e+00,  ..., 4.3428e-01,
          5.4074e-01, 5.4896e-01],
         [4.1160e+02, 4.1249e+02, 1.1993e+01,  ..., 5.5539e-01,
          4.5411e-01, 5.1596e-01],
         [4.1274e+02, 4.1125e+02, 3.7192e+01,  ..., 4.5205e-01,
          3.8751e-01, 4.5102e-01]]], device='cuda:0', grad_fn=<CatBackward0>)


In [ ]:
# import zipfile, os

# zip_path = "yolo/yolov3-weights.zip"
# extract_path = "yolo"

# if os.path.exists(zip_path):
#     with zipfile.ZipFile(zip_path, 'r') as zip_ref:
#         zip_ref.extractall(extract_path)
#     print("Dataset extracted to:", extract_path)
# else:
#     print("ZIP file not found.")

In [ ]:
# !curl -L -o yolo/yolov3-weights.zip\
#   https://www.kaggle.com/api/v1/datasets/download/shivam316/yolov3-weights

In [33]:
weight_path=os.path.join(base_dir, "yolo", "yolov3.weights")
model.load_weights(weight_path)

In [34]:
# inp = get_test_input()
# pred = model(inp)
# print (pred)

device = next(model.parameters()).device  # get model's device
inp = get_test_input().to(device)         # move input to same device
pred = model(inp)
print(pred)

tensor([[[8.5426e+00, 1.9015e+01, 1.1130e+02,  ..., 1.7306e-03,
          1.3874e-03, 9.2985e-04],
         [1.4105e+01, 1.8867e+01, 9.4014e+01,  ..., 5.9501e-04,
          9.2471e-04, 1.3085e-03],
         [2.1125e+01, 1.5269e+01, 3.5793e+02,  ..., 8.3609e-03,
          5.1067e-03, 5.8562e-03],
         ...,
         [4.1268e+02, 4.1069e+02, 3.7157e+00,  ..., 1.7185e-06,
          4.0955e-06, 6.5897e-07],
         [4.1132e+02, 4.1023e+02, 8.0353e+00,  ..., 1.3926e-05,
          3.2252e-05, 1.2076e-05],
         [4.1076e+02, 4.1318e+02, 4.9635e+01,  ..., 4.2174e-06,
          1.0794e-05, 1.8104e-05]]], device='cuda:0', grad_fn=<CatBackward0>)


In [35]:
write_results(pred.detach(), 0.5, 80, nms_conf = 0.4)

tensor([[  0.0000,  93.5403,  68.8597, 339.2717, 271.1132,   0.9469,   0.9985,
           1.0000],
        [  0.0000,  29.8484, 290.1097, 154.0395, 342.0089,   0.9992,   0.8164,
           7.0000],
        [  0.0000,   3.9401, 317.7851, 129.4821, 377.8968,   0.9947,   0.7065,
           7.0000],
        [  0.0000,  61.5692, 259.6971, 185.9224, 319.5284,   0.9018,   0.6905,
           7.0000],
        [  0.0000, 231.0338,   3.2243, 335.7471, 222.2702,   0.9999,   0.9936,
          16.0000]], device='cuda:0')

In [ ]:
# !mkdir -p yolo/data
# !wget -O yolo/data/coco.names https://raw.githubusercontent.com/ayooshkathuria/YOLO_v3_tutorial_from_scratch/master/data/coco.names


In [36]:
target_path = "yolo/data/coco.names"
source_url = "https://raw.githubusercontent.com/ayooshkathuria/YOLO_v3_tutorial_from_scratch/master/data/coco.names"
download_file(target_path, source_url)

In [37]:
def load_classes(namesfile):
    fp = open(namesfile, "r")
    names = fp.read().split("\n")[:-1]
    return names

In [38]:
num_classes = 80
classes = load_classes("yolo/data/coco.names")
print(classes)

['person', 'bicycle', 'car', 'motorbike', 'aeroplane', 'bus', 'train', 'truck', 'boat', 'traffic light', 'fire hydrant', 'stop sign', 'parking meter', 'bench', 'bird', 'cat', 'dog', 'horse', 'sheep', 'cow', 'elephant', 'bear', 'zebra', 'giraffe', 'backpack', 'umbrella', 'handbag', 'tie', 'suitcase', 'frisbee', 'skis', 'snowboard', 'sports ball', 'kite', 'baseball bat', 'baseball glove', 'skateboard', 'surfboard', 'tennis racket', 'bottle', 'wine glass', 'cup', 'fork', 'knife', 'spoon', 'bowl', 'banana', 'apple', 'sandwich', 'orange', 'broccoli', 'carrot', 'hot dog', 'pizza', 'donut', 'cake', 'chair', 'sofa', 'pottedplant', 'bed', 'diningtable', 'toilet', 'tvmonitor', 'laptop', 'mouse', 'remote', 'keyboard', 'cell phone', 'microwave', 'oven', 'toaster', 'sink', 'refrigerator', 'book', 'clock', 'vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']


In [ ]:
# !wget -O yolo/detect.py https://raw.githubusercontent.com/ayooshkathuria/YOLO_v3_tutorial_from_scratch/master/detect.py

In [39]:
target_path = "yolo/detect.py"
source_url = "https://raw.githubusercontent.com/ayooshkathuria/YOLO_v3_tutorial_from_scratch/master/detect.py"
download_file(target_path, source_url)

In [45]:
# !mkdir -p yolo/cocoimages
# !cp yolo/dog-cycle-car.png yolo/cocoimages/

cp: cannot create regular file 'yolo/cocoimages/': Not a directory


In [46]:
target_path = "yolo/cocoimages/dog-cycle-car.png"
source_url = "https://github.com/ayooshkathuria/pytorch-yolo-v3/raw/master/dog-cycle-car.png"
download_file(target_path, source_url)

In [41]:
# import sys
# import os

# yolo_path = os.path.abspath("yolo")
# if yolo_path not in sys.path:
#     sys.path.insert(0, yolo_path)

In [42]:
# from __future__ import division
# import time
# import torch
# import torch.nn as nn
# from torch.autograd import Variable
# import numpy as np
# import cv2
# from util import *
# import argparse
# import os
# import os.path as osp
# import pickle as pkl
# import pandas as pd
# import random

# import sys


# yolo_path = os.path.abspath("yolo")
# if yolo_path not in sys.path:
#     sys.path.insert(0, yolo_path)

# import darknet

# from darknet import Darknet


# images = "yolo/cocoimages"
# valid_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')

# batch_size = 4
# confidence = 0.5
# nms_thesh = 0.4
# start = 0
# CUDA = torch.cuda.is_available()

# num_classes = 80
# classes = load_classes("yolo/data/coco.names")

# #Set up the neural network

# print("Loading network.....")
# model = MyDarknet("yolo/cfg/yolov3.cfg")
# model.load_weights("yolo/yolov3.weights")
# print("Network successfully loaded")

# model.net_info["height"] = 416
# inp_dim = int(model.net_info["height"])
# assert inp_dim % 32 == 0
# assert inp_dim > 32

# #If there's a GPU availible, put the model on GPU

# if CUDA:
#     model.cuda()

# # Set the model in evaluation mode

# model.eval()

# read_dir = time.time()

# # Detection phase

# try:
#     imlist = [osp.join(osp.realpath('.'), images, img) for img in os.listdir(images)
#               if img.lower().endswith(valid_exts) and not img.startswith('.')]
#     print(imlist)
# except NotADirectoryError:
#     imlist = []
#     imlist.append(osp.join(osp.realpath('.'), images))

# except FileNotFoundError:
#     print ("No file or directory with the name {}".format(images))
#     exit()

# if not os.path.exists("des"):
#     os.makedirs("des")

# load_batch = time.time()
# loaded_ims = [cv2.imread(x) for x in imlist]

# im_batches = list(map(prep_image, loaded_ims, [inp_dim for x in range(len(imlist))]))
# im_dim_list = [(x.shape[1], x.shape[0]) for x in loaded_ims]
# im_dim_list = torch.FloatTensor(im_dim_list).repeat(1,2)


# leftover = 0
# if (len(im_dim_list) % batch_size):
#     leftover = 1

# if batch_size != 1:
#     num_batches = len(imlist) // batch_size + leftover
#     im_batches = [torch.cat((im_batches[i*batch_size : min((i +  1)*batch_size,
#                         len(im_batches))]))  for i in range(num_batches)]

# write = 0

# if CUDA:
#     im_dim_list = im_dim_list.cuda()

# start_det_loop = time.time()
# for i, batch in enumerate(im_batches):
#     # Load the image
#     start = time.time()
#     if CUDA:
#         batch = batch.cuda()
#     with torch.no_grad():
#         prediction = model(Variable(batch), CUDA)

#     prediction = write_results(prediction, confidence, num_classes, nms_conf = nms_thesh)

#     end = time.time()

#     if type(prediction) == int:

#         for im_num, image in enumerate(imlist[i*batch_size: min((i +  1)*batch_size, len(imlist))]):
#             im_id = i*batch_size + im_num
#             print("{0:20s} predicted in {1:6.3f} seconds".format(image.split("/")[-1], (end - start)/batch_size))
#             print("{0:20s} {1:s}".format("Objects Detected:", ""))
#             print("----------------------------------------------------------")
#         continue

#     prediction[:,0] += i*batch_size    #transform the atribute from index in batch to index in imlist

#     if not write:                      #If we have't initialised output
#         output = prediction
#         write = 1
#     else:
#         output = torch.cat((output,prediction))

#     for im_num, image in enumerate(imlist[i*batch_size: min((i +  1)*batch_size, len(imlist))]):
#         im_id = i*batch_size + im_num
#         objs = [classes[int(x[-1])] for x in output if int(x[0]) == im_id]
#         print("{0:20s} predicted in {1:6.3f} seconds".format(image.split("/")[-1], (end - start)/batch_size))
#         print("{0:20s} {1:s}".format("Objects Detected:", " ".join(objs)))
#         print("----------------------------------------------------------")

#     if CUDA:
#         torch.cuda.synchronize()
# try:
#     output
# except NameError:
#     print ("No detections were made")
#     exit()

# im_dim_list = torch.index_select(im_dim_list, 0, output[:,0].long())

# scaling_factor = torch.min(416/im_dim_list,1)[0].view(-1,1)

# output[:,[1,3]] -= (inp_dim - scaling_factor*im_dim_list[:,0].view(-1,1))/2
# output[:,[2,4]] -= (inp_dim - scaling_factor*im_dim_list[:,1].view(-1,1))/2

# output[:,1:5] /= scaling_factor

# for i in range(output.shape[0]):
#     output[i, [1,3]] = torch.clamp(output[i, [1,3]], 0.0, im_dim_list[i,0])
#     output[i, [2,4]] = torch.clamp(output[i, [2,4]], 0.0, im_dim_list[i,1])

# output_recast = time.time()
# class_load = time.time()
# colors = [[255, 0, 0], [255, 0, 0], [255, 255, 0], [0, 255, 0], [0, 255, 255], [0, 0, 255], [255, 0, 255]]

# draw = time.time()

# def write(x, results):
#     c1 = tuple(x[1:3].int())
#     c2 = tuple(x[3:5].int())
#     c1 = tuple(map(int, c1))
#     c2 = tuple(map(int, c2))
#     img = results[int(x[0])]
#     cls = int(x[-1])
#     color = random.choice(colors)
#     label = "{0}".format(classes[cls])
#     cv2.rectangle(img, c1, c2,color, 1)
#     t_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_PLAIN, 1 , 1)[0]
#     c2 = c1[0] + t_size[0] + 3, c1[1] + t_size[1] + 4
#     cv2.rectangle(img, c1, c2,color, -1)
#     cv2.putText(img, label, (c1[0], c1[1] + t_size[1] + 4), cv2.FONT_HERSHEY_PLAIN, 1, [225,255,255], 1);
#     return img


# list(map(lambda x: write(x, loaded_ims), output))

# det_names = pd.Series(imlist).apply(lambda x: "{}/det_{}".format("des",x.split("/")[-1]))

# list(map(cv2.imwrite, det_names, loaded_ims))

# end = time.time()

# print("SUMMARY")
# print("----------------------------------------------------------")
# print("{:25s}: {}".format("Task", "Time Taken (in seconds)"))
# print()
# print("{:25s}: {:2.3f}".format("Reading addresses", load_batch - read_dir))
# print("{:25s}: {:2.3f}".format("Loading batch", start_det_loop - load_batch))
# print("{:25s}: {:2.3f}".format("Detection (" + str(len(imlist)) +  " images)", output_recast - start_det_loop))
# print("{:25s}: {:2.3f}".format("Output Processing", class_load - output_recast))
# print("{:25s}: {:2.3f}".format("Drawing Boxes", end - draw))
# print("{:25s}: {:2.3f}".format("Average time_per_img", (end - load_batch)/len(imlist)))
# print("----------------------------------------------------------")


# torch.cuda.empty_cache()

In [43]:
def run_yolo_detection(
    model: nn.Module,
    image_dir: str,
    cfg_path: str,
    weights_path: str,
    class_path: str,
    output_dir: str = "des",
    batch_size: int = 4,
    confidence: float = 0.5,
    nms_thresh: float = 0.4,
    input_dim: int = 416
):
    import os, cv2, time, torch
    import os.path as osp
    import pandas as pd
    import random
    from util import load_classes, prep_image, write_results

    # Setup
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model.load_weights(weights_path)
    model.net_info["height"] = input_dim
    model.to(device).eval()
    classes = load_classes(class_path)
    colors = [[255, 0, 0], [255, 255, 0], [0, 255, 0], [0, 255, 255], [0, 0, 255], [255, 0, 255]]

    # Load images
    valid_exts = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')
    imlist = [osp.join(image_dir, img) for img in os.listdir(image_dir) if img.lower().endswith(valid_exts)]
    loaded_ims = [cv2.imread(img_path) for img_path in imlist]
    im_batches = list(map(prep_image, loaded_ims, [input_dim] * len(imlist)))
    im_dim_list = torch.FloatTensor([(img.shape[1], img.shape[0]) for img in loaded_ims]).repeat(1, 2).to(device)

    # Batch images
    leftover = len(imlist) % batch_size
    num_batches = len(imlist) // batch_size + bool(leftover)
    im_batches = [torch.cat(im_batches[i * batch_size : min((i + 1) * batch_size, len(im_batches))]) for i in range(num_batches)]

    # Detection loop
    output = None
    for i, batch in enumerate(im_batches):
        batch = batch.to(device)
        with torch.no_grad():
            prediction = model(batch)
        prediction = write_results(prediction, confidence, len(classes), nms_conf=nms_thresh)

        if isinstance(prediction, int):
            continue

        prediction[:, 0] += i * batch_size
        output = prediction if output is None else torch.cat((output, prediction))

    if output is None:
        print("No detections were made.")
        return

    # Post-processing
    im_dim_list = torch.index_select(im_dim_list, 0, output[:, 0].long())
    scaling_factor = torch.min(input_dim / im_dim_list, 1)[0].view(-1, 1)
    output[:, [1, 3]] -= (input_dim - scaling_factor * im_dim_list[:, 0].view(-1, 1)) / 2
    output[:, [2, 4]] -= (input_dim - scaling_factor * im_dim_list[:, 1].view(-1, 1)) / 2
    output[:, 1:5] /= scaling_factor

    for i in range(output.shape[0]):
        output[i, [1, 3]] = torch.clamp(output[i, [1, 3]], 0.0, im_dim_list[i, 0])
        output[i, [2, 4]] = torch.clamp(output[i, [2, 4]], 0.0, im_dim_list[i, 1])

    # Draw boxes
    def draw_box(x, results):
        c1 = tuple(map(int, x[1:3]))
        c2 = tuple(map(int, x[3:5]))
        img = results[int(x[0])]
        cls = int(x[-1])
        color = random.choice(colors)
        label = classes[cls]
        cv2.rectangle(img, c1, c2, color, 1)
        t_size = cv2.getTextSize(label, cv2.FONT_HERSHEY_PLAIN, 1, 1)[0]
        c2 = c1[0] + t_size[0] + 3, c1[1] + t_size[1] + 4
        cv2.rectangle(img, c1, c2, color, -1)
        cv2.putText(img, label, (c1[0], c1[1] + t_size[1] + 4), cv2.FONT_HERSHEY_PLAIN, 1, [225, 255, 255], 1)
        return img

    os.makedirs(output_dir, exist_ok=True)
    list(map(lambda x: draw_box(x, loaded_ims), output))
    det_names = pd.Series(imlist).apply(lambda x: f"{output_dir}/det_{osp.basename(x)}")
    list(map(cv2.imwrite, det_names, loaded_ims))

    print(f"Detections saved to '{output_dir}'")


In [47]:
model = MyDarknet("yolo/cfg/yolov3.cfg")
run_yolo_detection(
    model=model,
    image_dir="yolo/cocoimages",
    cfg_path="yolo/cfg/yolov3.cfg",
    weights_path="yolo/yolov3.weights",
    class_path="yolo/data/coco.names"
)

Detections saved to 'des'


## CoCo Dataset

In [57]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [58]:
import os
import requests
from tqdm import tqdm

def download_big_file(url, dest_path):
    if os.path.exists(dest_path):
        print(f" Skipping {os.path.basename(dest_path)} (already exists)")
        return
    response = requests.get(url, stream=True)
    total = int(response.headers.get('content-length', 0))
    with open(dest_path, 'wb') as file, tqdm(
        desc=os.path.basename(dest_path),
        total=total,
        unit='B',
        unit_scale=True,
        unit_divisor=1024,
    ) as bar:
        for data in response.iter_content(chunk_size=1024):
            size = file.write(data)
            bar.update(size)

#  COCO 2017 URLs
urls = {
    "train_images": "http://images.cocodataset.org/zips/train2017.zip",
    "val_images": "http://images.cocodataset.org/zips/val2017.zip",
    "annotations": "http://images.cocodataset.org/annotations/annotations_trainval2017.zip"
}

# 📁 Destination folder
dest_dir = "/content/drive/MyDrive/DLCV_Lab_Dataset/COCO/raw_zips"
os.makedirs(dest_dir, exist_ok=True)

#  Download all files
for name, url in urls.items():
    dest_path = os.path.join(dest_dir, url.split("/")[-1])
    download_big_file(url, dest_path)



train2017.zip: 100%|██████████| 18.0G/18.0G [21:10<00:00, 15.2MB/s]
val2017.zip: 100%|██████████| 778M/778M [05:13<00:00, 2.61MB/s]
annotations_trainval2017.zip: 100%|██████████| 241M/241M [00:24<00:00, 10.1MB/s]


In [64]:
!rm -rf /content/drive/MyDrive/DLCV_Lab_Dataset/COCO/val2017

In [65]:
import os
import zipfile
from tqdm import tqdm

def unzip_file(zip_path, extract_to):
    if os.path.exists(extract_to) and any(os.scandir(extract_to)):
        print(f"⚠️ Skipping extraction to {extract_to} (already populated)")
        return

    print(f"📦 Extracting {os.path.basename(zip_path)} → {extract_to}")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        members = zip_ref.infolist()
        for member in tqdm(members, desc="Extracting", unit="file"):
            zip_ref.extract(member, extract_to)

def unzip_folder(zip_path, extract_to, required_subdir="train"):
    expected_path = os.path.join(extract_to, required_subdir)
    if os.path.exists(expected_path) and any(os.scandir(expected_path)):
        print(f"⚠️ Skipping extraction: {expected_path} already populated")
        return

    print(f"📦 Extracting {os.path.basename(zip_path)} → {extract_to}")
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        for member in tqdm(zip_ref.infolist(), desc="Extracting", unit="file"):
            zip_ref.extract(member, extract_to)



In [66]:
#  Paths
raw_zip_dir = "/content/drive/MyDrive/DLCV_Lab_Dataset/COCO/raw_zips"
target_root = "/content/drive/MyDrive/DLCV_Lab_Dataset/COCO"

# Mapping zip files to target folders
# zip_targets = {
#     "train2017.zip": os.path.join(target_root, "train2017"),
#     "val2017.zip": os.path.join(target_root, "val2017"),
#     "annotations_trainval2017.zip": os.path.join(target_root, "annotations")
# }
zip_targets = {
    "train2017.zip": target_root,
    "val2017.zip": target_root,
    "annotations_trainval2017.zip": target_root
}

# Unzip each folder
for zip_name, target_dir in zip_targets.items():
    zip_path = os.path.join(raw_zip_dir, zip_name)
    required_subdir = os.path.splitext(zip_name)[0]  # removes .zip
    unzip_folder(zip_path, target_dir, required_subdir=required_subdir)

📦 Extracting train2017.zip → /content/drive/MyDrive/DLCV_Lab_Dataset/COCO


Extracting: 100%|██████████| 118288/118288 [1:36:36<00:00, 20.41file/s]


📦 Extracting val2017.zip → /content/drive/MyDrive/DLCV_Lab_Dataset/COCO


Extracting: 100%|██████████| 5001/5001 [02:41<00:00, 30.88file/s]


📦 Extracting annotations_trainval2017.zip → /content/drive/MyDrive/DLCV_Lab_Dataset/COCO


Extracting: 100%|██████████| 6/6 [00:17<00:00,  2.96s/file]


In [67]:
import shutil

def flatten_folder(src_root, expected_subfolder):
    sub_path = os.path.join(src_root, expected_subfolder)
    if os.path.exists(sub_path):
        for file in os.listdir(sub_path):
            shutil.move(os.path.join(sub_path, file), src_root)
        os.rmdir(sub_path)

In [68]:
# flatten_folder(os.path.join(target_root, "train2017"), "train2017")
# flatten_folder(os.path.join(target_root, "val2017"), "val2017")

In [69]:
path2data_train="/content/drive/MyDrive/DLCV_Lab_Dataset/COCO/train2017/"
path2json_train="/content/drive/MyDrive/DLCV_Lab_Dataset/COCO/annotations/instances_train2017.json"

path2data_val="/content/drive/MyDrive/DLCV_Lab_Dataset/COCO/val2017/"
path2json_val="/content/drive/MyDrive/DLCV_Lab_Dataset/COCO/annotations/instances_val2017.json"

img_size = 416

In [70]:
# coding=utf-8
import os
import sys
from PIL import Image
import torch
import math
import cv2
import numpy as np
import torch
import random
from torchvision.datasets import CocoDetection

from typing import Any, Callable, Optional, Tuple
import json

ANCHORS = [
            [[12, 16], [19, 36], [40, 28]],
            [[36, 75], [76, 55], [72, 146]],
            [[142, 110], [192, 243], [459, 401]]
]

STRIDES = [8, 16, 32]

IP_SIZE = img_size
NUM_ANCHORS = 3
NUM_CLASSES = 80

# with open('coco_cats.json') as js:
#     data = json.load(js)["categories"]
with open(path2json_train) as js:
    data = json.load(js)["categories"]

cats_dict = {cat["id"]: cat["name"] for cat in data}
print(cats_dict)



cats_dict = {}
for i in range(0, 80):
    cats_dict[str(data[i]['id'])] = i



class CustomCoco(CocoDetection):
    def __init__(
            self,
            root: str,
            annFile: str,
            transform: Optional[Callable] = None,
            target_transform: Optional[Callable] = None,
            transforms: Optional[Callable] = None,
    ) -> None:
        super(CocoDetection, self).__init__(root, transforms, transform, target_transform)
        from pycocotools.coco import COCO
        self.coco = COCO(annFile)
        self.ids = list(sorted(self.coco.imgs.keys()))


    def __getitem__(self, index: int) -> Tuple[Any, Any]:
        """
        Args:
            index (int): Index

        Returns:
            tuple: Tuple (image, target). target is the object returned by ``coco.loadAnns``.
        """
        coco = self.coco
        img_id = self.ids[index]
        ann_ids = coco.getAnnIds(imgIds=img_id)
        target = coco.loadAnns(ann_ids)

        path = coco.loadImgs(img_id)[0]['file_name']

        img = Image.open(os.path.join(self.root, path)).convert('RGB')
        img = np.array(img)

        category_ids = list(obj['category_id'] for obj in target)
        bboxes = list(obj['bbox'] for obj in target)

        if self.transform is not None:
            bboxes = list(obj['bbox'] for obj in target)
            category_ids = list(obj['category_id'] for obj in target)
            transformed = self.transform(image=img, bboxes=bboxes, category_ids=category_ids)
            img = transformed['image'],
            bboxes = torch.Tensor(transformed['bboxes'])
            cat_ids = torch.Tensor(transformed['category_ids'])
            labels, bboxes = self.__create_label(bboxes, cat_ids.type(torch.IntTensor))

        return img, labels, bboxes

    def __len__(self) -> int:
        return len(self.ids)

    def __create_label(self, bboxes, class_inds):
        """
        Label assignment. For a single picture all GT box bboxes are assigned anchor.
        1、Select a bbox in order, convert its coordinates("xyxy") to "xywh"; and scale bbox'
           xywh by the strides.
        2、Calculate the iou between the each detection layer'anchors and the bbox in turn, and select the largest
            anchor to predict the bbox.If the ious of all detection layers are smaller than 0.3, select the largest
            of all detection layers' anchors to predict the bbox.
        Note :
        1、The same GT may be assigned to multiple anchors. And the anchors may be on the same or different layer.
        2、The total number of bboxes may be more than it is, because the same GT may be assigned to multiple layers
        of detection.
        """
        # print("Class indices: ", class_inds)
        # bboxes = np.array(bboxes)
        # class_inds = np.array(class_inds)
        bboxes = np.array(bboxes,copy=True)
        class_inds = np.array(class_inds,copy=True)
        anchors = ANCHORS # all the anchors
        strides = np.array(STRIDES) # list of strides
        train_output_size = IP_SIZE / strides # image with different scales
        anchors_per_scale = NUM_ANCHORS # anchor per scale

        label = [
            np.zeros(
                (
                    int(train_output_size[i]),
                    int(train_output_size[i]),
                    anchors_per_scale,
                    5 + NUM_CLASSES,
                )
            )
            for i in range(3)
        ]
        # 150 bounding box ground truths per scale
        bboxes_xywh = [
            np.zeros((150, 4)) for _ in range(3)
        ]  # Darknet the max_num is 30
        bbox_count = np.zeros((3,))

        for i in range(len(bboxes)):
            bbox_coor = bboxes[i][:4]
            bbox_class_ind = cats_dict[str(class_inds[i])]

            # onehot
            one_hot = np.zeros(NUM_CLASSES, dtype=np.float32)
            one_hot[bbox_class_ind] = 1.0
            # one_hot_smooth = dataAug.LabelSmooth()(one_hot, self.num_classes)

            # convert "xyxy" to "xywh"
            bbox_xywh = np.concatenate(
                [
                    (0.5 * bbox_coor[2:] + bbox_coor[:2]) ,
                    bbox_coor[2:],
                ],
                axis=-1,
            )

            bbox_xywh_scaled = (
                1.0 * bbox_xywh[np.newaxis, :] / strides[:, np.newaxis]
            )

            iou = []
            exist_positive = False
            for i in range(3):
                anchors_xywh = np.zeros((anchors_per_scale, 4))
                anchors_xywh[:, 0:2] = (
                    np.floor(bbox_xywh_scaled[i, 0:2]).astype(np.int32) + 0.5
                )  # 0.5 for compensation

                # assign all anchors
                anchors_xywh[:, 2:4] = anchors[i]

                iou_scale = iou_xywh_numpy(
                    bbox_xywh_scaled[i][np.newaxis, :], anchors_xywh
                )
                iou.append(iou_scale)
                iou_mask = iou_scale > 0.3

                if np.any(iou_mask):
                    xind, yind = np.floor(bbox_xywh_scaled[i, 0:2]).astype(
                        np.int32
                    )

                    label[i][yind, xind, iou_mask, 0:4] = bbox_xywh * strides[i]
                    label[i][yind, xind, iou_mask, 4:5] = 1.0
                    label[i][yind, xind, iou_mask, 5:] = one_hot

                    bbox_ind = int(bbox_count[i] % 150)  # BUG : 150为一个先验值,内存消耗大
                    bboxes_xywh[i][bbox_ind, :4] = bbox_xywh * strides[i]
                    bbox_count[i] += 1

                    exist_positive = True

            if not exist_positive:
                # check if a ground truth bb have the best anchor with any scale
                best_anchor_ind = np.argmax(np.array(iou).reshape(-1), axis=-1)
                best_detect = int(best_anchor_ind / anchors_per_scale)
                best_anchor = int(best_anchor_ind % anchors_per_scale)

                xind, yind = np.floor(
                    bbox_xywh_scaled[best_detect, 0:2]
                ).astype(np.int32)

                label[best_detect][yind, xind, best_anchor, 0:4] = bbox_xywh * strides[best_detect]
                label[best_detect][yind, xind, best_anchor, 4:5] = 1.0
                # label[best_detect][yind, xind, best_anchor, 5:6] = bbox_mix
                label[best_detect][yind, xind, best_anchor, 5:] = one_hot

                bbox_ind = int(bbox_count[best_detect] % 150)
                bboxes_xywh[best_detect][bbox_ind, :4] = bbox_xywh * strides[best_detect]
                bbox_count[best_detect] += 1

        flatten_size_s = int(train_output_size[2]) * int(train_output_size[2]) * anchors_per_scale
        flatten_size_m = int(train_output_size[1]) * int(train_output_size[1]) * anchors_per_scale
        flatten_size_l = int(train_output_size[0]) * int(train_output_size[0]) * anchors_per_scale

        label_s = torch.Tensor(label[2]).view(1, flatten_size_s, 5 + NUM_CLASSES).squeeze(0)
        label_m = torch.Tensor(label[1]).view(1, flatten_size_m, 5 + NUM_CLASSES).squeeze(0)
        label_l = torch.Tensor(label[0]).view(1, flatten_size_l, 5 + NUM_CLASSES).squeeze(0)

        bboxes_s = torch.Tensor(bboxes_xywh[2])
        bboxes_m = torch.Tensor(bboxes_xywh[1])
        bboxes_l = torch.Tensor(bboxes_xywh[0])

        # label_sbbox, label_mbbox, label_lbbox = label
        sbboxes, mbboxes, lbboxes = bboxes_xywh
        # print("label")
        labels = torch.cat([label_l, label_m, label_s], 0)
        bboxes = torch.cat([bboxes_l, bboxes_m, bboxes_s], 0)
        return labels, bboxes

{1: 'person', 2: 'bicycle', 3: 'car', 4: 'motorcycle', 5: 'airplane', 6: 'bus', 7: 'train', 8: 'truck', 9: 'boat', 10: 'traffic light', 11: 'fire hydrant', 13: 'stop sign', 14: 'parking meter', 15: 'bench', 16: 'bird', 17: 'cat', 18: 'dog', 19: 'horse', 20: 'sheep', 21: 'cow', 22: 'elephant', 23: 'bear', 24: 'zebra', 25: 'giraffe', 27: 'backpack', 28: 'umbrella', 31: 'handbag', 32: 'tie', 33: 'suitcase', 34: 'frisbee', 35: 'skis', 36: 'snowboard', 37: 'sports ball', 38: 'kite', 39: 'baseball bat', 40: 'baseball glove', 41: 'skateboard', 42: 'surfboard', 43: 'tennis racket', 44: 'bottle', 46: 'wine glass', 47: 'cup', 48: 'fork', 49: 'knife', 50: 'spoon', 51: 'bowl', 52: 'banana', 53: 'apple', 54: 'sandwich', 55: 'orange', 56: 'broccoli', 57: 'carrot', 58: 'hot dog', 59: 'pizza', 60: 'donut', 61: 'cake', 62: 'chair', 63: 'couch', 64: 'potted plant', 65: 'bed', 67: 'dining table', 70: 'toilet', 72: 'tv', 73: 'laptop', 74: 'mouse', 75: 'remote', 76: 'keyboard', 77: 'cell phone', 78: 'micro

In [71]:
def iou_xywh_numpy(boxes1, boxes2):
    boxes1 = np.array(boxes1)
    boxes2 = np.array(boxes2)
    # print(boxes1, boxes2)

    boxes1_area = boxes1[..., 2] * boxes1[..., 3]
    boxes2_area = boxes2[..., 2] * boxes2[..., 3]

    boxes1 = np.concatenate([boxes1[..., :2] - boxes1[..., 2:] * 0.5,
                                boxes1[..., :2] + boxes1[..., 2:] * 0.5], axis=-1)
    boxes2 = np.concatenate([boxes2[..., :2] - boxes2[..., 2:] * 0.5,
                                boxes2[..., :2] + boxes2[..., 2:] * 0.5], axis=-1)

    left_up = np.maximum(boxes1[..., :2], boxes2[..., :2])
    right_down = np.minimum(boxes1[..., 2:], boxes2[..., 2:])

    inter_section = np.maximum(right_down - left_up, 0.0)
    inter_area = inter_section[..., 0] * inter_section[..., 1]
    union_area = boxes1_area + boxes2_area - inter_area
    IOU = 1.0 * inter_area / union_area
    return IOU


In [83]:
import torch
import math

def CIOU_xywh_torch(boxes1, boxes2):
    # Convert cx, cy, w, h → x1, y1, x2, y2
    boxes1 = torch.cat([boxes1[..., :2] - boxes1[..., 2:] * 0.5,
                        boxes1[..., :2] + boxes1[..., 2:] * 0.5], dim=-1)
    boxes2 = torch.cat([boxes2[..., :2] - boxes2[..., 2:] * 0.5,
                        boxes2[..., :2] + boxes2[..., 2:] * 0.5], dim=-1)

    # Ensure x1 < x2 and y1 < y2
    boxes1 = torch.cat([torch.min(boxes1[..., :2], boxes1[..., 2:]),
                        torch.max(boxes1[..., :2], boxes1[..., 2:])], dim=-1)
    boxes2 = torch.cat([torch.min(boxes2[..., :2], boxes2[..., 2:]),
                        torch.max(boxes2[..., :2], boxes2[..., 2:])], dim=-1)

    # Area
    boxes1_area = torch.clamp((boxes1[..., 2] - boxes1[..., 0]), min=1e-6) * \
                  torch.clamp((boxes1[..., 3] - boxes1[..., 1]), min=1e-6)
    boxes2_area = torch.clamp((boxes2[..., 2] - boxes2[..., 0]), min=1e-6) * \
                  torch.clamp((boxes2[..., 3] - boxes2[..., 1]), min=1e-6)

    # Intersection
    inter_left_up = torch.max(boxes1[..., :2], boxes2[..., :2])
    inter_right_down = torch.min(boxes1[..., 2:], boxes2[..., 2:])
    inter_section = torch.clamp(inter_right_down - inter_left_up, min=0)
    inter_area = inter_section[..., 0] * inter_section[..., 1]
    union_area = boxes1_area + boxes2_area - inter_area
    ious = inter_area / torch.clamp(union_area, min=1e-6)

    # Enclosing box
    outer_left_up = torch.min(boxes1[..., :2], boxes2[..., :2])
    outer_right_down = torch.max(boxes1[..., 2:], boxes2[..., 2:])
    outer = torch.clamp(outer_right_down - outer_left_up, min=0)
    outer_diagonal_line = torch.pow(outer[..., 0], 2) + torch.pow(outer[..., 1], 2)
    outer_diagonal_line = torch.clamp(outer_diagonal_line, min=1e-6)

    # Center distance
    boxes1_center = (boxes1[..., :2] + boxes1[..., 2:]) * 0.5
    boxes2_center = (boxes2[..., :2] + boxes2[..., 2:]) * 0.5
    center_dis = torch.pow(boxes1_center[..., 0] - boxes2_center[..., 0], 2) + \
                 torch.pow(boxes1_center[..., 1] - boxes2_center[..., 1], 2)

    # Aspect ratio penalty
    boxes1_size = torch.clamp(boxes1[..., 2:] - boxes1[..., :2], min=1e-6)
    boxes2_size = torch.clamp(boxes2[..., 2:] - boxes2[..., :2], min=1e-6)
    v = (4 / (math.pi ** 2)) * torch.pow(
        torch.atan(boxes1_size[..., 0] / boxes1_size[..., 1]) -
        torch.atan(boxes2_size[..., 0] / boxes2_size[..., 1]), 2)

    alpha = v / torch.clamp(1 - ious + v, min=1e-6)

    # Final CIoU
    cious = ious - (center_dis / outer_diagonal_line + alpha * v)
    cious = torch.clamp(cious, min=-1.0, max=1.0)  # Optional: keep CIoU bounded

    return cious

In [72]:
from __future__ import division
import time
import os
import os.path as osp
import numpy as np
import cv2
import pickle as pkl
import pandas as pd
import random
from copy import copy, deepcopy
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.autograd import Variable
from torch.utils.data import Subset
import torch.optim as optim
import torch.nn.functional as F

import torchvision
from torchvision import datasets, models, transforms

from util import *

import albumentations as A

In [73]:
train_transform = A.Compose([
    #A.SmallestMaxSize(256),
    A.Resize(img_size, img_size),
    # A.RandomCrop(width=224, height=224),
    # A.HorizontalFlip(p=0.5),
    # A.RandomBrightnessContrast(p=0.2),
], bbox_params=A.BboxParams(format='coco', label_fields=['category_ids']),
)

eval_transform = A.Compose([
    A.Resize(img_size, img_size),
    #A.SmallestMaxSize(256),
    #A.CenterCrop(width=224, height=224),
], bbox_params=A.BboxParams(format='coco', label_fields=['category_ids']),
)

In [74]:
import gc

# Clean up memory before training
gc.collect()                      # Python garbage collection
torch.cuda.empty_cache()         # Clears unused GPU memory
torch.cuda.reset_peak_memory_stats()  # Optional: resets memory tracking

In [75]:
BATCH_SIZE = 1
def collate_fn(batch):
    return tuple(zip(*batch))

train_dataset = Subset(CustomCoco(root = path2data_train,
                                annFile = path2json_train, transform=train_transform), list(range(0,20)))
train_dataloader = torch.utils.data.DataLoader(train_dataset, batch_size=BATCH_SIZE,
                                               shuffle=True, num_workers=0, collate_fn=collate_fn)

loading annotations into memory...
Done (t=24.70s)
creating index...
index created!


In [76]:
print("Loading network.....")
model = MyDarknet(cfg_path)
# load pretrained
# model.load_weights("yolov3.weights")
print("Network successfully loaded")

Loading network.....
Network successfully loaded


In [84]:
from torch.cuda.amp import autocast, GradScaler


def run_training(
    model,
    optimizer,
    dataloader,
    device,
    img_size,
    n_epoch,
    save_every_batch=False,
    save_every_epoch=True,
    ckpt_dir="checkpoints/"
):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    os.makedirs(ckpt_dir, exist_ok=True)
    scaler = GradScaler()

    for epoch_i in range(n_epoch):
        model.train()
        running_loss = 0.0

        for batch_i, (inputs, labels, bboxes) in enumerate(dataloader):
            inputs = torch.tensor(np.array(inputs), dtype=torch.float32).squeeze(1).permute(0, 3, 1, 2).to(device)
            labels = torch.stack(labels).to(device)

            optimizer.zero_grad()

            with autocast():
                outputs = model(inputs)

                pred_xywh = outputs[..., 0:4] / img_size
                pred_conf = outputs[..., 4:5]
                pred_cls = outputs[..., 5:]

                label_xywh = labels[..., :4] / img_size
                label_obj_mask = labels[..., 4:5]
                label_noobj_mask = 1.0 - label_obj_mask
                label_cls = labels[..., 5:]

                lambda_coord = 0.001
                lambda_noobj = 0.05

                loss_mse = nn.MSELoss(reduction='none')
                loss_bce_logits = nn.BCEWithLogitsLoss(reduction='none')  # ✅ Safe for autocast

                loss_coord = lambda_coord * label_obj_mask * loss_mse(pred_xywh, label_xywh)
                loss_conf = label_obj_mask * loss_bce_logits(pred_conf, label_obj_mask) + \
                            lambda_noobj * label_noobj_mask * loss_bce_logits(pred_conf, label_obj_mask)
                loss_cls = label_obj_mask * loss_bce_logits(pred_cls, label_cls)

                loss_coord = torch.sum(loss_coord)
                loss_conf = torch.sum(loss_conf)
                loss_cls = torch.sum(loss_cls)

                ciou = CIOU_xywh_torch(pred_xywh, label_xywh).unsqueeze(-1)
                loss_ciou = torch.sum(label_obj_mask * (1.0 - ciou))

                total_loss = loss_ciou + loss_conf + loss_cls

            scaler.scale(total_loss).backward()
            scaler.step(optimizer)
            scaler.update()

            running_loss += total_loss.item() * inputs.size(0)

            if save_every_batch:
                ckpt_path = os.path.join(ckpt_dir, f"epoch{epoch_i}_batch{batch_i}.pt")
                torch.save(model.state_dict(), ckpt_path)

            del inputs, labels, outputs, total_loss
            torch.cuda.empty_cache()
            gc.collect()

        epoch_loss = running_loss / len(dataloader.dataset)
        print(f"Epoch {epoch_i+1}/{n_epoch} - Loss: {epoch_loss:.4f}")

        if save_every_epoch:
            ckpt_path = os.path.join(ckpt_dir, f"epoch{epoch_i}.pt")
            torch.save(model.state_dict(), ckpt_path)

    print("✅ Training complete.")

In [85]:
model.to(device)

optimizer = optim.Adam(model.parameters(), lr=0.001)

n_epoch = 5
img_size = 416
save_every_batch = False
save_every_epoch = True
ckpt_dir = "checkpoints/"




In [86]:
print("Model device:", next(model.parameters()).device)
# print("Input device:", inputs.device)
print(f"Allocated: {torch.cuda.memory_allocated() / 1024**2:.2f} MB")
print(f"Reserved: {torch.cuda.memory_reserved() / 1024**2:.2f} MB")

Model device: cuda:0
Allocated: 1877.74 MB
Reserved: 2030.00 MB


In [87]:
run_training(
    model=model,
    optimizer=optimizer,
    dataloader=train_dataloader,
    device=device,
    img_size=416,
    n_epoch=5,
    save_every_batch=False,
    save_every_epoch=True,
    ckpt_dir=ckpt_dir
)


/tmp/ipython-input-2561573340.py:20: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipython-input-2561573340.py:32: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


Epoch 1/5 - Loss: nan
Epoch 2/5 - Loss: nan
Epoch 3/5 - Loss: nan
Epoch 4/5 - Loss: nan
Epoch 5/5 - Loss: nan
✅ Training complete.
